# Telecom Customer Churn Analysis and Prediction
**Bharat Goyal — BharatCares × AICTE × IBM SkillsBuild Data Analytics with AI Academic Internship**

### Aim
Analyze telecom customer information, identify patterns associated with churn, and build an interpretable binary classification model.

## Objectives
1. Inspect and clean the dataset.
2. Perform exploratory data analysis.
3. Visualize important churn patterns.
4. Prepare mixed numerical/categorical features.
5. Train Logistic Regression.
6. Evaluate accuracy, precision, recall, F1-score and ROC-AUC.
7. Interpret results and prepare business recommendations.

In [ ]:
import os
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
RANDOM_STATE=42
LOCAL_FILE="Telco-Customer-Churn.csv"
IBM_RAW_URL="https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
print("Libraries imported.")

In [ ]:
# Load dataset
if os.path.exists(LOCAL_FILE):
    df=pd.read_csv(LOCAL_FILE)
    print("Loaded local CSV.")
else:
    try:
        urllib.request.urlretrieve(IBM_RAW_URL, LOCAL_FILE)
        df=pd.read_csv(LOCAL_FILE)
        print("Downloaded IBM dataset.")
    except Exception as e:
        raise RuntimeError("Download failed. Put Telco-Customer-Churn.csv beside this notebook.") from e

print("Shape:",df.shape)
display(df.head())

## 1. Dataset Inspection

In [ ]:
print("Shape:",df.shape)
display(df.dtypes.to_frame("dtype"))
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
print("Duplicate customer IDs:",df["customerID"].duplicated().sum())

In [ ]:
df["TotalCharges"]=pd.to_numeric(df["TotalCharges"],errors="coerce")
df["ChurnFlag"]=df["Churn"].map({"No":0,"Yes":1})
print("Missing TotalCharges:",df["TotalCharges"].isna().sum())
display(df["Churn"].value_counts().to_frame("count"))
display((df["Churn"].value_counts(normalize=True)*100).round(2).to_frame("percentage"))

## 2. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(7,5))
sns.countplot(data=df,x="Churn")
plt.title("Customer Churn Distribution")
plt.xlabel("Churn"); plt.ylabel("Number of Customers")
plt.tight_layout(); plt.show()

In [ ]:
contract_rate=pd.crosstab(df["Contract"],df["Churn"],normalize="index")*100
display(contract_rate.round(2))
contract_rate.plot(kind="bar",figsize=(9,5))
plt.title("Churn Percentage by Contract Type")
plt.ylabel("Percentage"); plt.xlabel("Contract"); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

In [ ]:
internet_rate=pd.crosstab(df["InternetService"],df["Churn"],normalize="index")*100
display(internet_rate.round(2))
internet_rate.plot(kind="bar",figsize=(9,5))
plt.title("Churn Percentage by Internet Service")
plt.ylabel("Percentage"); plt.xlabel("Internet Service"); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(13,5))
sns.boxplot(data=df,x="Churn",y="tenure",ax=ax[0])
ax[0].set_title("Tenure by Churn")
sns.boxplot(data=df,x="Churn",y="MonthlyCharges",ax=ax[1])
ax[1].set_title("Monthly Charges by Churn")
plt.tight_layout(); plt.show()

In [ ]:
payment_rate=pd.crosstab(df["PaymentMethod"],df["Churn"],normalize="index")*100
display(payment_rate.round(2))
payment_rate.plot(kind="bar",figsize=(11,5))
plt.title("Churn Percentage by Payment Method")
plt.ylabel("Percentage"); plt.xlabel("Payment Method")
plt.xticks(rotation=25,ha="right")
plt.tight_layout(); plt.show()

## 3. Model Preparation

In [ ]:
df_model=df.dropna(subset=["ChurnFlag"]).copy()
X=df_model.drop(columns=["Churn","ChurnFlag","customerID"])
y=df_model["ChurnFlag"].astype(int)

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.20,random_state=RANDOM_STATE,stratify=y
)
print("Train:",X_train.shape,"Test:",X_test.shape)
print("Train churn rate:",round(y_train.mean()*100,2),"%")
print("Test churn rate:",round(y_test.mean()*100,2),"%")

In [ ]:
numeric_features=X_train.select_dtypes(include=["int64","float64"]).columns.tolist()
categorical_features=X_train.select_dtypes(include=["object"]).columns.tolist()

numeric_pipeline=Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler())
])
categorical_pipeline=Pipeline([
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("onehot",OneHotEncoder(handle_unknown="ignore",drop="first"))
])
preprocessor=ColumnTransformer([
    ("num",numeric_pipeline,numeric_features),
    ("cat",categorical_pipeline,categorical_features)
])
print("Numeric features:",numeric_features)
print("Categorical features:",len(categorical_features))

## 4. Logistic Regression

In [ ]:
model=Pipeline([
    ("preprocessor",preprocessor),
    ("classifier",LogisticRegression(max_iter=1000,random_state=RANDOM_STATE))
])
model.fit(X_train,y_train)
y_pred=model.predict(X_test)
y_prob=model.predict_proba(X_test)[:,1]
print("Training completed.")

## 5. Evaluation

In [ ]:
accuracy=accuracy_score(y_test,y_pred)
precision=precision_score(y_test,y_pred,zero_division=0)
recall=recall_score(y_test,y_pred,zero_division=0)
f1=f1_score(y_test,y_pred,zero_division=0)
roc_auc=roc_auc_score(y_test,y_prob)

metrics=pd.DataFrame({
    "Metric":["Accuracy","Precision","Recall","F1-score","ROC-AUC"],
    "Value":[accuracy,precision,recall,f1,roc_auc]
})
metrics["Value"]=metrics["Value"].round(4)
display(metrics)
print(classification_report(y_test,y_pred,target_names=["No Churn","Churn"],zero_division=0))

In [ ]:
cm=confusion_matrix(y_test,y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",
            xticklabels=["No Churn","Churn"],yticklabels=["No Churn","Churn"])
plt.title("Confusion Matrix"); plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.tight_layout(); plt.show()
print(cm)

In [ ]:
fpr,tpr,_=roc_curve(y_test,y_prob)
plt.figure(figsize=(7,5))
plt.plot(fpr,tpr,label=f"Logistic Regression (AUC={roc_auc:.3f})")
plt.plot([0,1],[0,1],"--",label="Random baseline")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve"); plt.legend(); plt.tight_layout(); plt.show()

## 6. Model Interpretation

In [ ]:
prep=model.named_steps["preprocessor"]
clf=model.named_steps["classifier"]
feature_names=prep.get_feature_names_out()
coef=clf.coef_[0]
coef_table=pd.DataFrame({"Feature":feature_names,"Coefficient":coef})
print("Largest positive coefficients:")
display(coef_table.sort_values("Coefficient",ascending=False).head(10))
print("Largest negative coefficients:")
display(coef_table.sort_values("Coefficient",ascending=True).head(10))

## 7. Business Recommendations
- Strengthen onboarding for newer customers.
- Investigate contract groups with elevated churn.
- Review high monthly-charge customers for price/service concerns.
- Improve technical support and complaint resolution.
- Use predicted probability to prioritize retention resources.
- Measure retention interventions rather than assuming they work.
- Keep human oversight for important customer decisions.

**Interpretation note:** `Churn` is an observed historical outcome. Predictions should be treated as risk estimates, not certainty.

## 8. Final Summary
The notebook demonstrates the complete workflow: data loading → cleaning → EDA → preprocessing → model training → evaluation → interpretation → recommendations.

For the project report, use the generated metric table, confusion matrix, ROC curve and EDA charts from the executed notebook.